# Llama 3.2 (3B) Baseline Evaluation
This notebook implements the execution plan to establish a baseline for medical dialogue summarization using Llama 3.2 (3B) with 4-bit quantization.

The goal is to use Zero-Shot and One-Shot prompt engineering to establish ROUGE-score baselines for Version 1.

## Step 1: Environment & Dependency Setup
Configuring the environment to handle 4-bit quantization, required for fitting a 3B parameter model into the RTX 3060 VRAM.

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets rouge-score huggingface_hub


## Step 2: Model Loading with 4-bit Quantization
Initialize `Llama-3.2-3B-Instruct` with `load_in_4bit=True` and `bnb_4bit_compute_dtype=torch.bfloat16`.
*Note: We are explicitly passing the Hugging Face token to circumvent gating restrictions.*

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"
hf_token = "YOUR_HF_TOKEN_HERE" # Provided access token

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading tokenizer {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

# Llama models do not have a default padding token
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model {model_id} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
print("Model loaded successfully.")


## Step 3: Data Loading & Formatting
Load the `omi-health/medical-dialogue-to-soap-summary` dataset. We define the system prompt and format it using the official Llama 3.2 Instruct format.

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("omi-health/medical-dialogue-to-soap-summary")

system_prompt = "You are a professional medical scribe. Your task is to summarize the following doctor-patient dialogue into a structured SOAP note (Subjective, Objective, Assessment, Plan)."

def format_prompt(dialogue, example=None):
    # Base system prompt
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system_prompt}<|eot_id|>\n"
    
    # If one-shot example is provided
    if example:
        prompt += f"<|start_header_id|>user<|end_header_id|>\n{example['dialogue']}<|eot_id|>\n"
        prompt += f"<|start_header_id|>assistant<|end_header_id|>\n{example['soap']}<|eot_id|>\n"
        
    # Target dialogue
    prompt += f"<|start_header_id|>user<|end_header_id|>\n{dialogue}<|eot_id|>\n"
    prompt += f"<|start_header_id|>assistant<|end_header_id|>\n"
    return prompt

print(f"Dataset loaded. Keys available: {list(dataset.keys())}")
print(f"Train split size: {len(dataset['train'])}")


## Step 4: Inference Execution
Execute Zero-Shot and One-Shot tests to establish the baseline. Llama 3.2 has a 128k context window, so no truncation is needed for our dialogues.

In [ ]:
from tqdm import tqdm

def generate_summary(prompt, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False # Greedy decoding for baseline reproducibility
        )
    
    # Decode only the newly generated tokens
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response.strip()

# For a baseline, we'll evaluate on a subset of the test/train data. 
# You can scale `num_eval_samples` up to evaluate the whole dataset.
eval_subset = dataset["train"].select(range(50)) 

one_shot_example = eval_subset[0] # Example for one-shot
eval_samples = eval_subset.select(range(1, min(101, len(eval_subset)))) # Rest for evaluation

results = []

print(f"Running Inference on {len(eval_samples)} samples...")
for sample in tqdm(eval_samples):
    # Ensure correct keys based on dataset structure
    dialogue = sample.get('dialogue', sample.get('Dialogue', ''))
    gt_soap = sample.get('soap', sample.get('Summary', ''))
    
    # 1. Zero-Shot
    prompt_zero = format_prompt(dialogue)
    pred_zero = generate_summary(prompt_zero)
    
    # 2. One-Shot
    example_dict = {
        'dialogue': one_shot_example.get('dialogue', one_shot_example.get('Dialogue', '')),
        'soap': one_shot_example.get('soap', one_shot_example.get('Summary', ''))
    }
    prompt_one = format_prompt(dialogue, example=example_dict)
    pred_one = generate_summary(prompt_one)
    
    results.append({
        "Dialogue": dialogue,
        "Ground Truth (SOAP)": gt_soap,
        "Llama_Prediction_ZeroShot": pred_zero,
        "Llama_Prediction_OneShot": pred_one
    })


## Step 5: Logging & Quantitative Evaluation
Calculate ROUGE scores for both sets of predictions and export the final baseline CSV.

In [ ]:
import pandas as pd
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

df = pd.DataFrame(results)

# Calculate ROUGE scores for Zero-Shot
rouge_l_zero = []
for _, row in df.iterrows():
    scores = scorer.score(row['Ground Truth (SOAP)'], row['Llama_Prediction_ZeroShot'])
    rouge_l_zero.append(scores['rougeL'].fmeasure)
df['ROUGE-L Score_ZeroShot'] = rouge_l_zero

# Calculate ROUGE scores for One-Shot
rouge_l_one = []
for _, row in df.iterrows():
    scores = scorer.score(row['Ground Truth (SOAP)'], row['Llama_Prediction_OneShot'])
    rouge_l_one.append(scores['rougeL'].fmeasure)
df['ROUGE-L Score_OneShot'] = rouge_l_one

print(f"Average Zero-Shot ROUGE-L: {df['ROUGE-L Score_ZeroShot'].mean():.4f}")
print(f"Average One-Shot ROUGE-L: {df['ROUGE-L Score_OneShot'].mean():.4f}")

# For the requested v1_llama_baseline.csv format, we will export the Zero-Shot as the primary baseline,
# but also include One-Shot for completeness.
export_df = pd.DataFrame({
    "Dialogue": df["Dialogue"],
    "Ground Truth (SOAP)": df["Ground Truth (SOAP)"],
    "Llama_Prediction": df["Llama_Prediction_ZeroShot"], # Outputting zero-shot as the baseline Llama prediction
    "ROUGE-L Score": df["ROUGE-L Score_ZeroShot"],
    "Llama_Prediction_OneShot": df["Llama_Prediction_OneShot"],
    "ROUGE-L Score_OneShot": df["ROUGE-L Score_OneShot"]
})

export_df.to_csv("v1_llama_baseline.csv", index=False)
print("Exported baseline metrics to 'v1_llama_baseline.csv'")
